In [2]:
# Optimized ABIDE Parcellation using Glasser+Tian (gt_atlas_map)
# Output: (150 timepoints × 414 ROIs) fMRI + standardized phenotype

import sys
import os

# Add gt_atlas directory to Python path so we can import gt_atlas_map
sys.path.append("/home/jaizor/jaizor/Ξ/gt_atlas")

import numpy as np
import pandas as pd
from pathlib import Path
from nilearn import datasets
from gt_atlas_map import GlasserTianParcellator, create_analysis_phenotype
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# =============================================================================
# CONFIGURATION
# =============================================================================
ATLAS_DIR = "/home/jaizor/jaizor/Ξ/gt_atlas"          # ✅ Correct atlas path
DATA_DIR = "/home/jaizor/jaizor/xtra/data/nilearn_data"
OUTPUT_DIR = Path("dataset/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ABIDE I TR mapping (official)
ABIDE_I_TR = {
    'Caltech': 2.0, 'CMU': 2.0, 'KKI': 1.5, 'MaxMun': 3.0,
    'NYU': 2.0, 'Olin': 1.5, 'SBL': 2.2, 'SDSU': 2.0,
    'Stanford': 2.0, 'Trinity': 2.0, 'UCLA': 3.0, 'UM': 2.0,
    'USM': 2.0, 'PITT': 1.5, 'Yale': 2.0, 'Utah': 2.0
}

TARGET_TR = 2.0
TARGET_DURATION = 300.0  # 5 minutes → 150 timepoints


# =============================================================================
# HELPER: Extract TRs from ABIDE phenotype
# =============================================================================
def prepare_phenotype(phenotypic_df):
    """Robustly extract TR using ABIDE site mapping."""
    site_col = None
    for col in phenotypic_df.columns:
        clean = col.upper().replace('_', '').replace('ID', '')
        if clean in ['SITE', 'SITEID']:
            site_col = col
            break
    if site_col is None:
        for col in phenotypic_df.columns:
            if 'site' in col.lower():
                site_col = col
                break
    if site_col is None:
        raise ValueError(f"SITE_ID not found. Columns: {list(phenotypic_df.columns)}")

    tr_values = phenotypic_df[site_col].map(ABIDE_I_TR)
    unmapped = tr_values.isna()
    if unmapped.any():
        logger.warning(f"⚠️ {unmapped.sum()} subjects have unmapped sites → using TR=2.0")
        tr_values = tr_values.fillna(2.0)
    if tr_values.isna().all():
        raise ValueError("All TR values are NaN!")

    tr_values = tr_values.astype(np.float32)
    phenotypic_df = phenotypic_df.copy()
    phenotypic_df['TR'] = tr_values
    return tr_values.values, phenotypic_df



# =============================================================================
# MAIN PIPELINE
# =============================================================================
def main():
    logger.info("📥 Loading ABIDE data...")
    abide = datasets.fetch_abide_pcp(
        data_dir=DATA_DIR,
        pipeline="cpac",
        derivatives=["func_preproc"],
        quality_checked=True,
        verbose=0
    )

    logger.info("🔍 Preparing phenotype and TR values...")
    tr_values, phenotype_full = prepare_phenotype(abide.phenotypic)

    logger.info("🚀 Parcellating with Glasser+Tian atlases...")
    parcellator = GlasserTianParcellator(ATLAS_DIR)
    processed_data, valid_indices = parcellator.process_dataset(
        fmri_paths=abide.func_preproc,
        tr_values=tr_values,
        n_jobs=-1,
        target_tr=TARGET_TR,
        target_duration=TARGET_DURATION
    )

    # Filter phenotype to valid subjects
    valid_phenotype = phenotype_full.iloc[valid_indices].reset_index(drop=True)

    # >>> ✅ CRITICAL: Save in the CORRECT format expected by your model pipeline <<<
    OUTPUT_DIR = Path("dataset/output")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # --- Save fMRI: (N, 150, 414) + subject_ids as strings ---
    data_array = np.stack(processed_data, axis=0).astype(np.float32)  # (N, T, R)

    # Convert to fixed-length Unicode string array (safe for npz)
    subject_ids = valid_phenotype['SUB_ID'].astype(str).values
    # Determine max length to avoid truncation
    max_len = max(len(sid) for sid in subject_ids)
    subject_ids_safe = np.array(subject_ids, dtype=f'U{max_len}')  # e.g., U6 for '50122'

    np.savez_compressed(
        OUTPUT_DIR / "fmri_ASD.npz",
        data=data_array,
        subject_ids=subject_ids_safe  # ✅ Now dtype=U6 (not object)
    )

    # --- Save phenotype with 'eid' as COLUMN (string) ---
    pheno_analysis = pd.DataFrame({
        'eid': valid_phenotype['SUB_ID'].astype(str),
        'Age': valid_phenotype['AGE_AT_SCAN'],
        'Sex': (valid_phenotype['SEX'] == 1).astype(int),   # Male=1
        'ASD': (valid_phenotype['DX_GROUP'] == 1).astype(int)  # ASD=1
    })
    pheno_analysis.to_csv(OUTPUT_DIR / "pheno_ASD.csv", index=False)

    # Final summary
    logger.info("\n" + "=" * 60)
    logger.info("🎉 ABIDE Parcellation Complete!")
    logger.info(f"✅ Processed: {len(processed_data)} / {len(abide.func_preproc)} subjects")
    logger.info(f"✅ Saved fMRI: {OUTPUT_DIR / 'fmri_ASD.npz'}")
    logger.info(f"✅ Saved phenotype: {OUTPUT_DIR / 'pheno_ASD.csv'}")
    logger.info(f"✅ Shape per subject: ({int(TARGET_DURATION / TARGET_TR)}, 414)")
    logger.info(f"✅ Duration: {TARGET_DURATION}s | TR: {TARGET_TR}s")
    logger.info("=" * 60)

    print("\n📊 Sample phenotype:")
    print(pheno_analysis.head(3))
    print(f"\nClass balance (ASD=1): {pheno_analysis['ASD'].value_counts().to_dict()}")

# =============================================================================
# RUN
# =============================================================================
if __name__ == "__main__":
    main()

2025-12-09 11:57:15,583 - INFO - 📥 Loading ABIDE data...
2025-12-09 11:57:15,656 - INFO - 🔍 Preparing phenotype and TR values...
2025-12-09 11:57:15,657 - WARNING - ⚠️ 485 subjects have unmapped sites → using TR=2.0
2025-12-09 11:57:15,659 - INFO - 🚀 Parcellating with Glasser+Tian atlases...
2025-12-09 11:57:15,660 - INFO - Processing 871 subjects with n_jobs=-1
2025-12-09 11:57:16,762 - WARNING - Atlases misaligned with fMRI data — will resample atlases
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 40 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 tasks      | elapsed:    3.9s
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    4.1s
[Parallel(n_jobs=-1)]: Done  33 tasks      | elapsed:    4.2s
[Parallel(n_jobs=-1)]: Done  48 tasks      | elapsed:    7.8s
[Parallel(n_jobs=-1)]: Done  65 tasks      | elapsed:    8.4s
[Parallel(n_jobs=-1)]: Done  82 tasks      | elapsed:    8.4s
[Parallel(n_jobs=-1)]: Done 101 tasks      | elapsed:   10.2s
[Parallel(n_jobs=-1)]: Done 12


📊 Sample phenotype:
     eid   Age  Sex  ASD
0  50102  14.0    1    0
1  50103  14.0    1    0
2  50104  16.0    1    0

Class balance (ASD=1): {0: 314, 1: 271}
